In [2]:
import os

In [3]:
%pwd

'c:\\Users\\2021ICTS28\\Desktop\\wine_quality_end_to_end_project\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\2021ICTS28\\Desktop\\wine_quality_end_to_end_project'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    

In [7]:
from src.wineProject.utils.common import read_yaml, create_directories
from src.wineProject.constants import *

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )
        
        return data_ingestion_config

In [9]:
import os
from urllib import request as requests
import zipfile
from src.wineProject import logger

In [10]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = requests.urlretrieve(
                self.config.source_URL, 
                self.config.local_data_file
            )
            logger.info(f"Data downloaded successfully to {self.config.local_data_file}")
        else:
            logger.info(f"Data already exists at {self.config.local_data_file}")
            
    def extract_zip_file(self):
            unzip_path = self.config.unzip_dir
            os.makedirs(unzip_path, exist_ok=True)
            
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)
            logger.info(f"Data extracted successfully to {unzip_path}")
            
            

In [11]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
    logger.info("Data ingestion completed successfully.")
    
except Exception as e:
    logger.exception(f"An error occurred during data ingestion: {e}")

[2026-03-19 16:50:05,094: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-03-19 16:50:05,102: INFO: common]: yaml file: params.yaml loaded successfully
[2026-03-19 16:50:05,104: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-03-19 16:50:05,104: INFO: common]: created directory at: artifacts
[2026-03-19 16:50:05,104: INFO: common]: created directory at: artifacts/data_ingestion
[2026-03-19 16:50:06,574: INFO: 1453832110]: Data downloaded successfully to artifacts/data_ingestion/data.zip
[2026-03-19 16:50:06,593: INFO: 1453832110]: Data extracted successfully to artifacts/data_ingestion
[2026-03-19 16:50:06,593: INFO: 1606906020]: Data ingestion completed successfully.
